# BP4 Gate 3 — Aggregation-Pipeline Benchmark & Champion Selection
**Customer360 Navigator Enterprise Suite — Customer Journey Analytics**

## Why this gate looks different from BP1/BP2/BP3's own Gate 3
CRISP-DM's "Modeling" phase maps to Gate 3 for every BP (Section 10.1) — but BP4 has no supervised
target (Master Plan Section 5.1/7: "do not invent customer IDs... if longitudinal identity is
absent, call this event/issue journey analytics explicitly"), so there is nothing to classify and
no champion-by-predictive-metric to select. Master Plan Section 17.5 names BP4's actual Gate 3
work explicitly: **a Polars/DuckDB lazy-aggregation-pipeline benchmark** ("Lazy scans, SQL-style
group-by push-down, no per-row Python loops"). This notebook benchmarks real candidate execution
engines on BP4's real group-by workload — the same two issue-cluster Gold tables Gate 2 already
built and real-run confirmed — and selects a champion **by real measured wall-clock speed**, never
by accuracy/F1/any predictive metric.

## Correctness before speed — the one non-negotiable rule
A fast-but-wrong candidate is not a contender. Every candidate is checked against Gate 2's own
real, already-confirmed Gold tables (`cfpb_issue_cluster_summary_gold.parquet`,
`cfpb_issue_cluster_monthly_gold.parquet`) **before** it is timed at all. Only candidates whose
output exactly matches (after normalizing dtype/row-order/float-representation differences that
are artifacts of the engine, not the data) enter the timing race. A candidate that produces the
wrong numbers is disqualified and reported as disqualified — never silently dropped, never timed
anyway.

## The 5 candidates
- **`pandas_groupby`** — pandas naive baseline (plays the same "basic/baseline" role
  `logistic_regression` played in BP3's own classifier benchmark).
- **`polars_eager`** — Polars eager `DataFrame.group_by()`.
- **`polars_lazy`** — Polars lazy `scan_parquet` + `group_by` + `collect()`. This is **Gate 2's
  own already-real-run-confirmed production implementation**, reused unmodified from
  `src/features/bp4_journey_features.py` (HYPER) — its presence in this race is also a
  self-consistency check: it must reproduce Gate 2's own ground truth exactly, or something has
  drifted since Gate 2 ran.
- **`polars_lazy_streaming`** — the same lazy plan, collected via Polars' out-of-core streaming
  engine (`collect(engine="streaming")`, with a fallback to the older `collect(streaming=True)`
  API for an older installed Polars, live-detected rather than assumed) — the WARP-relevant lever
  for a dataset too large to collect comfortably in RAM.
- **`duckdb_sql`** — DuckDB SQL `GROUP BY` push-down directly over the Parquet file, Section
  17.5's own named candidate. **Live-checked, not assumed**: `requirements.txt` lists
  `duckdb>=1.1`, but BP1 Gate 3's own real run on this exact machine found it **NOT INSTALLED**.
  This notebook re-checks live rather than trusting that finding as still true; if still missing,
  this one candidate is skipped with an explicit, honest note — it never blocks the other 4
  candidates, and a missing optional dependency is never fabricated as a result.

## What this notebook does, concretely
1. Confirms Gate 2 real-run prerequisites live (`configs/bp4_customer_journey_analytics.yaml`),
   then loads Gate 2's own real Gold tables as the correctness ground truth — never a separately
   invented expectation.
2. Runs every live-available candidate once on the real
   `data/processed/cfpb_journey_event_gold.parquet` (Gate 2's real row-level output — this isolates
   the aggregation/group-by step itself as the one variable under test, holding the input data
   identical across every candidate, mirroring this project's own "identical folds across every
   candidate" principle from the classifier-benchmark gates).
3. Normalizes and diffs every candidate's output against ground truth (tolerant float comparison,
   dtype-agnostic) — disqualifies (with a stated reason) any candidate that doesn't match exactly.
4. Times every correctness-passing candidate over 5 repeated real runs (one unmeasured warmup run
   first, `gc.collect()` before each timed run) and records real peak-RSS deltas via `psutil`
   (best-effort — process-level RSS is affected by OS page caching and Python's own GC, so this is
   reported as an approximate signal, never claimed as an exact allocation count).
5. Selects the champion — the correctness-passing candidate with the lowest real minimum
   wall-clock time — and states plainly whether it agrees with Gate 2's current production
   implementation (`polars_lazy`) or would be a Gate 6 recommendation to change it.
6. Writes the full per-candidate results table and the Gate 3 config block.

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; you run it. Every timing number below is a
  real measurement from your machine, not a simulated or assumed one.
- **Zero-fabrication**: a candidate that is not installed, errors, or produces incorrect output is
  reported as exactly that — never silently substituted, never given a fabricated timing.
- **WARP**: `configure_performance()` first, before any heavy import. This notebook additionally
  sets `POLARS_MAX_THREADS` explicitly (Polars reads its own thread-pool size from this env var,
  not `OMP_NUM_THREADS`/`MKL_NUM_THREADS` — `configure_performance()` does not set it) and pins
  DuckDB's own thread count to the identical WARP ceiling via `PRAGMA threads`, so every candidate
  races under the same thread budget — a fair benchmark, not fair by accident. Still the
  project-standing ≤92% CPU/RAM ceiling throughout, never 100%.
- **HYPER**: the `polars_lazy` and `polars_lazy_streaming` candidates call
  `build_issue_cluster_summary`/`build_issue_cluster_monthly` from `src/features/
  bp4_journey_features.py` directly, unmodified — no aggregation logic is reimplemented for those
  two candidates. Reuses `src/utils/bp1_config_sync.py` and the project's own
  try/`ImportError`-based live-package-check convention from `00_hardware_benchmark.ipynb`.
- **Idempotent**: re-running this notebook overwrites this gate's own config block and
  `gate3_benchmark_results.csv` in place; every other gate's block and Gate 1's front matter are
  preserved verbatim regardless of position.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same project-root resolver as every other notebook.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp4_customer_journey_analytics/artifacts/gate3_benchmark_results.csv` — one row per
  candidate: availability, correctness verdict, timing stats, memory delta, champion flag.
- `configs/bp4_customer_journey_analytics.yaml` — Gate 3 marker block appended/overwritten.

## Prerequisites
BP4 Gate 2 must have been real-run at least once — this notebook checks
`journey_row_count_matches_raw` and `cluster_count_matches_gate1` live in the config file and
raises a clear error if either is missing, rather than silently proceeding without a correctness
reference.

## If a structural check below fails
It raises `AssertionError` naming the failing check. A check failing because **zero** candidates
produced correct output is a genuine data-integrity problem and must never be worked around by
loosening the comparison tolerance. A missing optional dependency (DuckDB) is not a failure — it is
reported honestly and the benchmark proceeds on whatever candidates are actually available.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp4_customer_journey_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp4_customer_journey_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
BP4_CONFIG_PATH = CONFIGS_DIR / "bp4_customer_journey_analytics.yaml"
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import. Also sets
# POLARS_MAX_THREADS here - Polars reads its own thread-pool size from this env var (not
# OMP_NUM_THREADS/MKL_NUM_THREADS), and configure_performance() does not set it, so this notebook
# sets it explicitly at the same "before first import" timing configure_performance() itself
# requires, so every candidate engine below races under the identical WARP thread ceiling - a fair
# benchmark, not fair by accident.
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
os.environ["POLARS_MAX_THREADS"] = str(WARP_SUMMARY["n_threads_configured"])
print(
    f"[WARP] POLARS_MAX_THREADS set to {WARP_SUMMARY['n_threads_configured']} (same WARP ceiling as "
    "every other thread-pool env var above). This only takes effect if Polars has not already spun up "
    "its thread pool earlier in this kernel session - if a later section reports a different effective "
    "thread count, restart the kernel and re-run from Section 1 (LESSONS_LEARNED_APPLIED.md #12-style "
    "stale-kernel-state caveat)."
)
DUCKDB_THREADS = WARP_SUMMARY["n_threads_configured"]

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12) + live
# package-availability check (never assumed - reuses the try/import/except-ImportError pattern
# 00_hardware_benchmark.ipynb's own detect_library_versions() established for this project, HYPER).
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import gc  # noqa: E402
import importlib.util  # noqa: E402
import statistics  # noqa: E402
import time  # noqa: E402
import warnings  # noqa: E402

import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import psutil  # noqa: E402
import yaml  # noqa: E402

from features.bp4_journey_features import (  # noqa: E402
    CLUSTER_KEY,
    build_issue_cluster_monthly,
    build_issue_cluster_summary,
)
from utils.bp1_config_sync import write_gate_block  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)


def _is_installed(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


DUCKDB_INSTALLED = _is_installed("duckdb")
print(
    f"[OK] Live package-availability check: duckdb {'INSTALLED' if DUCKDB_INSTALLED else 'NOT INSTALLED'} "
    "(requirements.txt lists duckdb>=1.1, but BP1 Gate 3's own real run on this machine found it missing - "
    "re-checked live here rather than trusted as still true; if missing, that ONE candidate is skipped "
    "with an explicit note below, never fabricated)."
)

JOURNEY_EVENT_GOLD_PATH = DATA_PROCESSED_DIR / "cfpb_journey_event_gold.parquet"
GT_SUMMARY_PATH = DATA_PROCESSED_DIR / "cfpb_issue_cluster_summary_gold.parquet"
GT_MONTHLY_PATH = DATA_PROCESSED_DIR / "cfpb_issue_cluster_monthly_gold.parquet"
N_TIMING_REPEATS = 5

# ============================================================
# SECTION 4: Gate 2 prerequisite check (live) + load the real ground-truth Gold tables Gate 2
# already wrote - the benchmark's correctness reference is Gate 2's OWN real output, never a
# separately invented expectation.
# ============================================================
assert BP4_CONFIG_PATH.exists(), f"{BP4_CONFIG_PATH} does not exist - run Gate 1 first."
with open(BP4_CONFIG_PATH, "r", encoding="utf-8") as f:
    FULL_CONFIG = yaml.safe_load(f.read())
# The config file's marker lines ("# --- Gate N ... ---") are plain YAML comments, so the whole
# file - front matter plus every appended gate block - parses as one flat mapping; no custom
# front-matter/block splitting is needed here (bp1_config_sync.py's splitting logic is only needed
# for WRITING one block without disturbing the others, not for reading everything at once).
gate1_status_confirmed = "gate1_confirmed" in str(FULL_CONFIG.get("status", ""))
gate2_confirmed = (
    gate1_status_confirmed
    and FULL_CONFIG.get("journey_row_count_matches_raw") is True
    and FULL_CONFIG.get("cluster_count_matches_gate1") is True
)
assert gate2_confirmed, (
    "BP4 Gate 2 does not appear to have completed successfully (status="
    f"{FULL_CONFIG.get('status')!r}, journey_row_count_matches_raw="
    f"{FULL_CONFIG.get('journey_row_count_matches_raw')!r}, cluster_count_matches_gate1="
    f"{FULL_CONFIG.get('cluster_count_matches_gate1')!r}). Run Gate 2 for real before Gate 3."
)
print(
    f"[OK] Gate 2 prerequisite confirmed (journey_row_count={FULL_CONFIG.get('journey_row_count'):,}, "
    f"n_clusters={FULL_CONFIG.get('n_clusters'):,})."
)

assert JOURNEY_EVENT_GOLD_PATH.exists(), f"{JOURNEY_EVENT_GOLD_PATH} missing - run Gate 2 for real first."
assert GT_SUMMARY_PATH.exists(), f"{GT_SUMMARY_PATH} missing - run Gate 2 for real first."
assert GT_MONTHLY_PATH.exists(), f"{GT_MONTHLY_PATH} missing - run Gate 2 for real first."

gt_summary_df = pl.read_parquet(GT_SUMMARY_PATH).to_pandas()
gt_monthly_df = pl.read_parquet(GT_MONTHLY_PATH).to_pandas()
gt_summary_matches_gate2_block = len(gt_summary_df) == FULL_CONFIG.get("n_clusters")
gt_monthly_matches_gate2_block = len(gt_monthly_df) == FULL_CONFIG.get("cluster_monthly_row_count")
print(
    f"[OK] Real ground-truth Gold tables loaded live: summary={len(gt_summary_df):,} rows (matches "
    f"Gate 2's recorded n_clusters: {gt_summary_matches_gate2_block}), monthly={len(gt_monthly_df):,} "
    f"rows (matches Gate 2's recorded cluster_monthly_row_count: {gt_monthly_matches_gate2_block})."
)

# ============================================================
# SECTION 5: Normalization + frame-comparison helpers. Every candidate returns a plain pandas
# DataFrame (the lowest-common-denominator format every engine below can produce), which is then
# normalized to a directly comparable form - sorted rows/columns, string-cast dates/categoricals,
# rounded floats - before any equality check, since two engines that compute the identical real
# result can still disagree on dtype/row-order/float-representation without this.
# ============================================================


def normalize_for_compare(df: pd.DataFrame, key_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        series = out[col]
        if pd.api.types.is_bool_dtype(series):
            out[col] = series.astype(bool)
        elif pd.api.types.is_float_dtype(series):
            out[col] = series.astype(float).round(6)
        elif pd.api.types.is_integer_dtype(series):
            out[col] = series.astype("int64")
        else:
            # Covers real dates, categoricals/strings from every engine (Polars Categorical ->
            # pandas category, DuckDB VARCHAR, pandas object) - string is the one representation
            # every engine can be normalized to without inventing a shared typed schema.
            out[col] = series.astype(str)
    out = out.sort_values(by=key_cols).reset_index(drop=True)
    return out[sorted(out.columns)]


def frames_match(a: pd.DataFrame, b: pd.DataFrame) -> tuple[bool, str]:
    if sorted(a.columns) != sorted(b.columns):
        return False, f"column set mismatch: {sorted(set(a.columns) ^ set(b.columns))}"
    if len(a) != len(b):
        return False, f"row count mismatch: {len(a)} vs {len(b)}"
    try:
        pd.testing.assert_frame_equal(
            a.reset_index(drop=True),
            b.reset_index(drop=True),
            check_dtype=False,
            check_exact=False,
            atol=1e-6,
            rtol=1e-6,
        )
        return True, "OK"
    except AssertionError as exc:
        return False, str(exc)[:400]


gt_summary_norm = normalize_for_compare(gt_summary_df, CLUSTER_KEY)
gt_monthly_norm = normalize_for_compare(gt_monthly_df, CLUSTER_KEY + ["complaint_month"])

# ============================================================
# SECTION 6: The 5 candidate aggregation-pipeline implementations. Every candidate reads the SAME
# real input (JOURNEY_EVENT_GOLD_PATH, Gate 2's real row-level output) so the only variable under
# test is the execution engine/strategy, mirroring this project's own "identical folds/
# preprocessing across every candidate" principle from the classifier-benchmark gates.
# ============================================================


def run_pandas_groupby(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_parquet(path, engine="pyarrow")
    key_cols = list(CLUSTER_KEY)
    # Cast key columns to plain str before grouping - a deliberate defensive choice, not just
    # cosmetic: pandas groupby on categorical columns can silently include unobserved category
    # combinations unless observed=True is set; casting to str first plus observed=True below is
    # belt-and-suspenders against that real, documented pandas gotcha, not a hoped-for default.
    for col in key_cols:
        df[col] = df[col].astype(str)
    summary = (
        df.groupby(key_cols, dropna=False, observed=True)
        .agg(
            n_complaints_total=("Complaint ID", "count"),
            first_complaint_date=("date_received_parsed", "min"),
            last_complaint_date=("date_received_parsed", "max"),
            n_active_months=("complaint_month", "nunique"),
            avg_response_lag_days=("response_lag_days", "mean"),
            banking77_coverage_fraction=("banking77_in_scope", "mean"),
        )
        .reset_index()
    )
    summary["is_recurring_cluster"] = summary["n_complaints_total"] > 1
    monthly = (
        df.groupby(key_cols + ["complaint_month"], dropna=False, observed=True)
        .agg(
            n_complaints_month=("Complaint ID", "count"),
            avg_response_lag_days_month=("response_lag_days", "mean"),
            n_banking77_in_scope_month=("banking77_in_scope", "sum"),
        )
        .reset_index()
    )
    return summary, monthly


def run_polars_eager(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pl.read_parquet(path)
    summary = (
        df.group_by(CLUSTER_KEY)
        .agg(
            pl.len().alias("n_complaints_total"),
            pl.col("date_received_parsed").min().alias("first_complaint_date"),
            pl.col("date_received_parsed").max().alias("last_complaint_date"),
            pl.col("complaint_month").n_unique().alias("n_active_months"),
            pl.col("response_lag_days").mean().alias("avg_response_lag_days"),
            pl.col("banking77_in_scope").mean().alias("banking77_coverage_fraction"),
        )
        .with_columns((pl.col("n_complaints_total") > 1).alias("is_recurring_cluster"))
    )
    monthly = df.group_by(CLUSTER_KEY + ["complaint_month"]).agg(
        pl.len().alias("n_complaints_month"),
        pl.col("response_lag_days").mean().alias("avg_response_lag_days_month"),
        pl.col("banking77_in_scope").sum().alias("n_banking77_in_scope_month"),
    )
    return summary.to_pandas(), monthly.to_pandas()


def run_polars_lazy(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Gate 2's own already-real-run-confirmed production implementation, reused unmodified
    (HYPER) - its presence here is also a self-consistency check against Gate 2's own ground
    truth, not just another benchmark entry."""
    journey_lazy = pl.scan_parquet(path)
    summary = build_issue_cluster_summary(journey_lazy).collect()
    monthly = build_issue_cluster_monthly(journey_lazy).collect()
    return summary.to_pandas(), monthly.to_pandas()


def _collect_streaming(lazy_frame: "pl.LazyFrame") -> "pl.DataFrame":
    """Live-detects the correct streaming-collect API rather than assuming one Polars version -
    the modern engine="streaming" kwarg (matches this project's requirements.txt polars>=1.9 pin)
    is tried first, with a fallback to the older streaming=True kwarg for an older installed
    Polars."""
    try:
        return lazy_frame.collect(engine="streaming")
    except TypeError:
        return lazy_frame.collect(streaming=True)


def run_polars_lazy_streaming(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    journey_lazy = pl.scan_parquet(path)
    summary = _collect_streaming(build_issue_cluster_summary(journey_lazy))
    monthly = _collect_streaming(build_issue_cluster_monthly(journey_lazy))
    return summary.to_pandas(), monthly.to_pandas()


def run_duckdb_sql(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    import duckdb

    con = duckdb.connect(database=":memory:")
    try:
        con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
        key_cols_sql = ", ".join(f'"{c}"' for c in CLUSTER_KEY)
        summary = con.execute(
            f"""
            SELECT {key_cols_sql},
                   COUNT(*) AS n_complaints_total,
                   MIN(date_received_parsed) AS first_complaint_date,
                   MAX(date_received_parsed) AS last_complaint_date,
                   COUNT(DISTINCT complaint_month) AS n_active_months,
                   AVG(response_lag_days) AS avg_response_lag_days,
                   AVG(CAST(banking77_in_scope AS DOUBLE)) AS banking77_coverage_fraction
            FROM read_parquet(?)
            GROUP BY {key_cols_sql}
            """,
            [str(path)],
        ).df()
        summary["is_recurring_cluster"] = summary["n_complaints_total"] > 1
        monthly = con.execute(
            f"""
            SELECT {key_cols_sql}, complaint_month,
                   COUNT(*) AS n_complaints_month,
                   AVG(response_lag_days) AS avg_response_lag_days_month,
                   SUM(CAST(banking77_in_scope AS BIGINT)) AS n_banking77_in_scope_month
            FROM read_parquet(?)
            GROUP BY {key_cols_sql}, complaint_month
            """,
            [str(path)],
        ).df()
    finally:
        con.close()
    return summary, monthly


CANDIDATE_AVAILABILITY = {
    "pandas_groupby": _is_installed("pandas"),
    "polars_eager": _is_installed("polars"),
    "polars_lazy": _is_installed("polars"),
    "polars_lazy_streaming": _is_installed("polars"),
    "duckdb_sql": DUCKDB_INSTALLED,
}
CANDIDATES = {
    "pandas_groupby": run_pandas_groupby,
    "polars_eager": run_polars_eager,
    "polars_lazy": run_polars_lazy,
    "polars_lazy_streaming": run_polars_lazy_streaming,
    "duckdb_sql": run_duckdb_sql,
}

# ============================================================
# SECTION 7: Correctness verification pass - every live-available candidate is run ONCE and
# diffed against Gate 2's real ground truth BEFORE any timing happens. A candidate that errors or
# mismatches is disqualified and reported as exactly that - never silently dropped, never timed
# anyway.
# ============================================================
process = psutil.Process()
status_map: dict[str, str] = {}
correctness_reason: dict[str, str] = {}
correct_outputs: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}

for name in CANDIDATES:
    if not CANDIDATE_AVAILABILITY.get(name, False):
        status_map[name] = "NOT_INSTALLED"
        correctness_reason[name] = "module not installed on this machine (live-verified, never fabricated)"
        print(f"[SKIP] {name}: not installed")
        continue
    fn = CANDIDATES[name]
    try:
        summary_out, monthly_out = fn(JOURNEY_EVENT_GOLD_PATH)
    except Exception as exc:  # noqa: BLE001 - one candidate erroring must never crash the whole benchmark
        status_map[name] = "ERRORED"
        correctness_reason[name] = f"{type(exc).__name__}: {exc}"[:400]
        print(f"[ERROR] {name}: {correctness_reason[name]}")
        continue

    summary_norm = normalize_for_compare(summary_out, CLUSTER_KEY)
    monthly_norm = normalize_for_compare(monthly_out, CLUSTER_KEY + ["complaint_month"])
    summary_match, summary_reason = frames_match(summary_norm, gt_summary_norm)
    monthly_match, monthly_reason = frames_match(monthly_norm, gt_monthly_norm)
    is_correct = summary_match and monthly_match

    if is_correct:
        status_map[name] = "CORRECT"
        correctness_reason[name] = "OK"
        correct_outputs[name] = (summary_out, monthly_out)
    else:
        status_map[name] = "INCORRECT"
        correctness_reason[name] = f"summary={summary_reason}; monthly={monthly_reason}"
    print(
        f"[{'OK' if is_correct else 'MISMATCH'}] {name} correctness: summary_match={summary_match}, "
        f"monthly_match={monthly_match}"
    )

n_correct = sum(1 for s in status_map.values() if s == "CORRECT")
print(f"\n[OK] Correctness verification complete: {n_correct}/{len(CANDIDATES)} candidates correct.")

# ============================================================
# SECTION 8: Timing + memory benchmark pass - ONLY correctness-passing candidates race. One
# unmeasured warmup run first, then N_TIMING_REPEATS real timed runs with gc.collect() before
# each. Memory delta via psutil (best-effort/approximate - OS page caching and Python's own GC
# affect process RSS, so this is a signal, never claimed as an exact allocation count).
# ============================================================
timing_results: dict[str, dict] = {}

for name in correct_outputs:
    fn = CANDIDATES[name]
    fn(JOURNEY_EVENT_GOLD_PATH)  # warmup, unmeasured

    times: list[float] = []
    mem_deltas_mb: list[float] = []
    for _ in range(N_TIMING_REPEATS):
        gc.collect()
        mem_before = process.memory_info().rss
        t0 = time.perf_counter()
        fn(JOURNEY_EVENT_GOLD_PATH)
        t1 = time.perf_counter()
        mem_after = process.memory_info().rss
        times.append(t1 - t0)
        mem_deltas_mb.append((mem_after - mem_before) / (1024**2))

    mean_s = statistics.mean(times)
    min_s = min(times)
    std_s = statistics.pstdev(times) if len(times) > 1 else 0.0
    mean_mem_mb = statistics.mean(mem_deltas_mb)
    timing_results[name] = {
        "mean_seconds": round(mean_s, 6),
        "min_seconds": round(min_s, 6),
        "std_seconds": round(std_s, 6),
        "mean_mem_delta_mb": round(mean_mem_mb, 3),
    }
    print(
        f"[TIMED] {name}: mean={mean_s:.4f}s min={min_s:.4f}s std={std_s:.4f}s "
        f"mem~{mean_mem_mb:.1f}MB (n={N_TIMING_REPEATS} real repeats)"
    )

# ============================================================
# SECTION 9: Champion selection - fastest real minimum wall-clock time among correctness-passing
# candidates only. Never a champion-by-accuracy call; there is no predictive metric here.
# ============================================================
assert timing_results, (
    "No candidate produced correct output - cannot select a champion. This is a genuine gate "
    "failure (an aggregation-logic or environment problem), never something to work around by "
    "loosening the correctness comparison."
)

CHAMPION = min(timing_results, key=lambda n: timing_results[n]["min_seconds"])
champion_matches_current_production_implementation = CHAMPION == "polars_lazy"
print(f"\n[CHAMPION] {CHAMPION} (min_seconds={timing_results[CHAMPION]['min_seconds']:.4f})")
print(
    "[OK] Champion matches Gate 2's current production implementation (polars_lazy): "
    f"{champion_matches_current_production_implementation}"
)
if not champion_matches_current_production_implementation:
    print(
        f"[NOTE] Champion '{CHAMPION}' outperformed the current production implementation "
        "'polars_lazy' on this real run - recorded here as a Gate 6 (Productization) "
        "recommendation, not applied retroactively to Gate 2's already-real-run-confirmed output."
    )

# ============================================================
# SECTION 10: Write the real per-candidate benchmark results table (every candidate, including
# skipped/errored/incorrect ones - never only the winners).
# ============================================================
benchmark_rows = []
for name in CANDIDATES:
    row = {
        "candidate": name,
        "available": CANDIDATE_AVAILABILITY.get(name, False),
        "status": status_map.get(name, "NOT_RUN"),
        "correctness_note": correctness_reason.get(name, ""),
        "mean_seconds": timing_results.get(name, {}).get("mean_seconds"),
        "min_seconds": timing_results.get(name, {}).get("min_seconds"),
        "std_seconds": timing_results.get(name, {}).get("std_seconds"),
        "mean_mem_delta_mb": timing_results.get(name, {}).get("mean_mem_delta_mb"),
        "n_timing_repeats": N_TIMING_REPEATS if name in timing_results else None,
        "is_champion": name == CHAMPION,
    }
    benchmark_rows.append(row)

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_csv_path = ARTIFACTS_DIR / "gate3_benchmark_results.csv"
benchmark_df.to_csv(benchmark_csv_path, index=False)
print(f"\n[SAVED] {benchmark_csv_path.relative_to(PROJECT_ROOT)}")
print(benchmark_df.to_string(index=False))

# ============================================================
# SECTION 11: Write the Gate 3 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified, fourth BP4 gate to do so).
# ============================================================
gate3_marker = (
    "# --- Gate 3 (Aggregation-Pipeline Benchmark & Champion Selection) results "
    "(appended, idempotent overwrite) ---"
)
gate3_block_lines = (
    [
        f"candidates_total: {len(CANDIDATES)}",
        f"candidates_available: {sum(1 for v in CANDIDATE_AVAILABILITY.values() if v)}",
        f"candidates_correct: {n_correct}",
        "candidate_status:",
    ]
    + [f"  {name}: {status_map[name]}" for name in CANDIDATES]
    + [
        f'champion_pipeline: "{CHAMPION}"',
        f"champion_min_seconds: {timing_results[CHAMPION]['min_seconds']}",
        f"champion_mean_seconds: {timing_results[CHAMPION]['mean_seconds']}",
        "champion_matches_current_production_implementation: "
        f"{champion_matches_current_production_implementation}",
        f"n_timing_repeats: {N_TIMING_REPEATS}",
        f"duckdb_live_installed: {DUCKDB_INSTALLED}",
        f"polars_max_threads_configured: {WARP_SUMMARY['n_threads_configured']}",
        f'benchmark_results_path: "{benchmark_csv_path.relative_to(PROJECT_ROOT).as_posix()}"',
    ]
)
write_gate_block(BP4_CONFIG_PATH, gate3_marker, gate3_block_lines)
print(f"[SAVED] gate3 block written to {BP4_CONFIG_PATH.relative_to(PROJECT_ROOT)}")

with open(BP4_CONFIG_PATH, "r", encoding="utf-8") as f:
    _post_write_config_text = f.read()
gate3_block_actually_written = gate3_marker in _post_write_config_text

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "gate2_prerequisite_confirmed": gate2_confirmed,
    "ground_truth_gold_tables_loaded": len(gt_summary_df) > 0 and len(gt_monthly_df) > 0,
    "ground_truth_summary_matches_gate2_recorded_count": gt_summary_matches_gate2_block,
    "ground_truth_monthly_matches_gate2_recorded_count": gt_monthly_matches_gate2_block,
    "at_least_one_candidate_available": any(CANDIDATE_AVAILABILITY.values()),
    "at_least_one_candidate_correct": n_correct > 0,
    "polars_lazy_was_attempted": CANDIDATE_AVAILABILITY.get("polars_lazy", False),
    "polars_lazy_is_correct": status_map.get("polars_lazy") == "CORRECT",
    "champion_is_among_correct_candidates": CHAMPION in timing_results,
    "champion_has_lowest_min_seconds": timing_results[CHAMPION]["min_seconds"]
    == min(r["min_seconds"] for r in timing_results.values()),
    "benchmark_csv_written": benchmark_csv_path.exists(),
    "benchmark_csv_row_count_matches_candidate_count": len(benchmark_df) == len(CANDIDATES),
    "config_gate3_block_written": gate3_block_actually_written,
}

print("\n=== INTEGRITY CHECKS ===")
for check_name, passed in checks.items():
    result_label = "[PASS]" if passed else "[FAIL]"
    print(f"{result_label} {check_name}")
    assert passed, f"[CHECK FAILED] {check_name}"

print(
    f"\n[ALL CHECKS PASSED] BP4 Gate 3 complete - {n_correct}/{len(CANDIDATES)} candidates produced "
    f"correct output, champion='{CHAMPION}' "
    f"({'matches' if champion_matches_current_production_implementation else 'DIFFERS FROM'} Gate 2's "
    "current production implementation 'polars_lazy'). Proceed to BP4 Gate 4 next."
)
